<a href="https://colab.research.google.com/github/Nmg1994/ActiveTransportation/blob/main/ActiveTransportation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mesa

In [ ]:
from mesa import Model, Agent
from mesa.datacollection import DataCollector
from mesa.time import Schedule

import geopandas as gpd
import networkx as nx
import numpy as np
import random
from shapely.geometry import Point
from scipy.spatial import KDTree
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import minimize
import pandas as pd
from matplotlib.patches import Patch

# Model development

In [ ]:
class TransportModel(Model):

  def __init__(self, n_rep = 100):
    super().__init__()
    self.agent_list = []
    self.Each_agent_represents = n_rep
    self.montreal_da = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/MontrealDA.shp")
    self.schools = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/Schools.shp")
    self.residential = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/Residential.shp")
    self.workplaces = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/Workplaces.shp")
    self.bixi = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/Bixi_stations.shp")
    self.stm = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/STM.shp")
    self.nodes = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/Nodes.shp")
    self.edges = gpd.read_file("/content/drive/MyDrive/Transportation_UrbanHealth/Edges.shp")

    self.edges.loc[self.edges['vitesse'] < 10, 'vitesse'] = 10

    # Build Network
    self.G = nx.MultiDiGraph()

    # Add nodes
    for _, row in self.nodes.iterrows():
      self.G.add_node(int(row['ID']), x=row.geometry.x, y=row.geometry.y, has_bixi=False, has_stm=False, workplace_ids=[], school_ids=[])

    valid_nodes = set(self.G.nodes)

    # Add edges
    for _, row in self.edges.iterrows():

      from_id = int(row['F_node'])
      to_id = int(row['T_node'])

      if from_id == to_id:
        continue

      if from_id not in valid_nodes or to_id not in valid_nodes:
          continue

      ED_L = max(float(row['edge_lengt']), 0.0001) / 1000 # Km
      no_bike = int(row['no_bike']) # no_bike = 1 it means bike is not permitted
      no_carr = int(row['no_car']) # no_car = 1 it means car is not permitted
      no_pedestr = int(row['no_pedestr']) # no_pedestr = 1 it means pedestration is not permitted
      spd = float(row['vitesse'])
      trv_t = (ED_L / spd) * 60

      attrs = dict(
                edge_length=ED_L,
                no_bikelane=no_bike,
                no_car=no_carr,
                no_pedestrain=no_pedestr,
                speed=spd,
                travel_time=trv_t
            )

      if row['oneway'] == 'FT':
          self.G.add_edge(from_id, to_id, **attrs)

      elif row['oneway'] == 'TF':
          self.G.add_edge(to_id, from_id, **attrs)

      else:
          self.G.add_edge(from_id, to_id, **attrs)
          self.G.add_edge(to_id, from_id, **attrs)

    # Build KDTree
    self.node_ids = list(self.G.nodes)
    coords = np.array([(self.G.nodes[n]['x'], self.G.nodes[n]['y']) for n in self.node_ids])
    self.kdtree = KDTree(coords)

    # Attach infrastructure
    for _, wp in self.workplaces.iterrows():
        n = self.nearest_node(wp.geometry.centroid.x, wp.geometry.centroid.y)
        self.G.nodes[n]['workplace_ids'].append(wp['UniqueID'])

    for _, sch in self.schools.iterrows():
        n = self.nearest_node(sch.geometry.centroid.x, sch.geometry.centroid.y)
        self.G.nodes[n]['school_ids'].append(sch['UniqueID'])

    for _, b in self.bixi.iterrows():
        n = self.nearest_node(b.geometry.x, b.geometry.y)
        self.G.nodes[n]['has_bixi'] = True

    for _, s in self.stm.iterrows():
        n = self.nearest_node(s.geometry.x, s.geometry.y)
        self.G.nodes[n]['has_stm'] = True

    # Cache nodes
    self.workplace_nodes = [n for n, d in self.G.nodes(data=True) if d['workplace_ids']] #it includes the node_id
    self.school_nodes = [n for n, d in self.G.nodes(data=True) if d['school_ids']]
    self.bixi_nodes = [n for n, d in self.G.nodes(data=True) if d['has_bixi']]
    self.stm_nodes = [n for n, d in self.G.nodes(data=True) if d['has_stm']]

    # Create agents
    self.create_agents()

    # Data collection
    self.datacollector = DataCollector(
        agent_reporters={
            "DA_ID": "DA_ID",
            "age": "age",
            "income": "income",
            "dist_work": "dist_work",
            "dist_work_time": "dist_work_time",
            "dist_home_stm": "dist_home_stm",
            "dist_home_bixi": "dist_home_bixi",
            "dist_stm_work": "dist_stm_work",
            "dist_bixi_work": "dist_bixi_work",
            "dist_bixi_bixi": "dist_bixi_bixi",
            "dist_stm_stm": "dist_stm_stm",
            "bike_lane_ratio": "bike_lane_ratio",
            "bixi_density": "bixi_density",
            "mode_choice": "mode_choice"
        }
    )


  def car_time(self, u, v, d):
    if d.get("no_car", 0) == 1:
        return float("inf")
    return d.get("travel_time", float("inf"))

  def bike_distance(self, u, v, d):
    if d.get("no_bikelane", 0) == 1:
        return float("inf")
    return d.get("edge_length", float("inf"))

  def walk_distance(self, u, v, d):
    if d.get("no_pedestrain", 0) == 1:
        return float("inf")
    return d.get("edge_length", float("inf"))

  def weight_general(self, u, v, d):
    return d.get("edge_length", float("inf"))

  # --------------------------------------------------------
  def nearest_node(self, x, y):
    dist, idx = self.kdtree.query([x, y])
    return self.node_ids[idx]

  # --------------------------------------------------------
  def add_bixi_stations(self, num_stations): # Intervention of the increases in Bixi stations

    # Candidate nodes (no BIXI yet)
    candidates = [n for n, d in self.G.nodes(data=True) if not d['has_bixi']]

    if not candidates:
      print("No candidates to add Bixi stations.")
      return

    # Randomly select nodes
    selected = random.sample(candidates, min(num_stations, len(candidates)))

    # Adding the selected nodes to the network
    for n in selected:
      self.G.nodes[n]['has_bixi'] = True

    self.bixi_nodes = [n for n, d in self.G.nodes(data=True) if d['has_bixi']]

  # --------------------------------------------------------
  def add_bike_lanes(self, percentage):

    # Candidate edges (no bike lane)
    candidates = [(u, v, k) for u, v, k, d in self.G.edges(keys=True, data=True) if d.get('no_bikelane', 0) == 1]

    if not candidates:
      print("No candidates to add bike lanes.")
      return

    num_to_convert = int(len(candidates) * percentage)
    selected = random.sample(candidates, min(num_to_convert, len(candidates)))

    for u, v, k in selected:
      self.G[u][v][k]['no_bikelane'] = 0

  # --------------------------------------------------------
  def create_agents(self):

    pbar = tqdm(self.montreal_da.iterrows(), total=len(self.montreal_da))
    for _, da in pbar:
      pbar.set_description(f"Creating agents for DA: {da['DA_name']}")

      if da['Car_truck_'] == 0 and da['Public_tra'] == 0 and da['Bicycle'] == 0 and da['Walked'] == 0:
        continue

      else:
        # Residential polygons inside this DA
        res = self.residential[self.residential.intersects(da.geometry)]
        if res.empty:
          continue

        income_val = 0
        # Handle income with zero
        if da['Median_tot'] != 0:
            income_val = da['Median_tot']
        else:
            income_val = self.montreal_da[self.montreal_da['Median_tot'] > 0]['Median_tot'].median()


        # Initializing Children (0–14)
        num_children = int((da['F0_14_year'] / self.Each_agent_represents))

        for _ in range(num_children):

          poly = res.sample(1).geometry.values[0]
          pt = poly.centroid

          agent = PersonAgent(self, {
              'age': random.randint(0, 14),
              'age_group': 'child',
              'DA_ID': da['DA_name'],
              'income': 0,
              'median_age': da['Median_age'],
              'x': pt.x,
              'y': pt.y
          })

          self.agent_list.append(agent)

        # Initializing Adults (15–64)

        num_adults = int((da['F15__64_ye'] / self.Each_agent_represents))

        for _ in range(num_adults):

          poly = res.sample(1).geometry.values[0]
          pt = poly.centroid

          agent = PersonAgent(self, {
              'age': random.randint(15, 64),
              'age_group': 'adult',
              'DA_ID': da['DA_name'],
              'income': income_val,
              'median_age': da['Median_age'],
              'x': pt.x,
              'y': pt.y
          })

          self.agent_list.append(agent)

  def compute_mode_shares(self): # This function is defined to be used only in model calibration

    results = {}

    for _, row in self.montreal_da.iterrows():

      if row['Car_truck_'] == 0 and row['Public_tra'] == 0 and row['Bicycle'] == 0 and row['Walked'] == 0: # Opting out the DA with no people using any four transportation modes (i.e., outliers)
        continue

      res = self.residential[self.residential.intersects(row.geometry)]
      if res.empty:
        continue

      da_id = row['DA_name']

      agentss = [a for a in self.agent_list if a.age_group == "adult" and a.DA_ID == da_id]

      counts = {'auto':0, 'transit':0, 'bike':0, 'walk':0}

      for a in agentss:
        if a.mode_choice == "auto": counts['auto'] += 1
        elif a.mode_choice == "transit": counts['transit'] += 1
        elif a.mode_choice == "bike": counts['bike'] += 1
        else: counts['walk'] += 1

        results[da_id] = counts.copy()

    return results

  def step(self):
    # Activate agents in random order
    for agentt in tqdm(random.sample(self.agent_list, len(self.agent_list)), desc= "Agents activation"):
        agentt.step()

    self.datacollector.collect(self)

In [ ]:
class PersonAgent(Agent):

  def __init__(self, model, attrs):
    super().__init__(model)

    # Demographic
    self.age = attrs['age']
    self.age_group = attrs['age_group']
    self.DA_ID = attrs['DA_ID']
    self.income = attrs.get('income', None)

    # Location
    self.x = attrs['x']
    self.y = attrs['y']
    self.home_node = None

    # Destinations
    self.workplace_node = None
    self.school_node = None
    self.home_bixi_node = None
    self.home_stm_node = None
    self.work_bixi_node = None
    self.work_stm_node = None

    # Distances
    self.dist_work = None # the distance from home to the workplace/school
    self.dist_work_time = None # the time takes to go the distance from home to the workplace/school by car
    self.dist_home_bixi = None  # distance from home to the nearest Bixi station
    self.dist_home_stm = None   # distance from home to the nearest STM station

    self.dist_bixi_work = None # distance from the nearest Bixi station to workplace
    self.dist_stm_work = None  # distance from the nearest STM station to workplace

    self.dist_bixi_bixi = None # distance from the home-nearest Bixi station to the work-nearest Bixi station
    self.dist_stm_stm = None   # distance from the home-nearest STM station to the work-nearest STM station

    # Policy related characteristics
    self.bike_lane_ratio = 0
    self.bixi_density = 0

    # Utilities
    self.U_auto = 0
    self.U_transit = 0
    self.U_bike = 0
    self.U_walk = 0

    # Probabilities
    self.P_auto = 0
    self.P_transit = 0
    self.P_bike = 0
    self.P_walk = 0

    # Mode
    self.mode_choice = None

    # ---------------- Speed cache ----------------
    self._cached_home = None
    self._cache_lengths = None
    self._cache_precomputed = False
    self._cache_lengths_home_walk = None
    self._cashe_lengths_workplace_walk = None
    self._cache_precomputed_walk = False

  # --------------------------------------------------------
  def step(self):
    self.initialize()
    self.Bixi_STM_settings()

  # --------------------------------------------------------

  def _get_home_lengths(self):

    if self._cache_precomputed:
      return self._cache_lengths

    self.home_node = self.model.nearest_node(self.x, self.y)

    self._cache_lengths = nx.single_source_dijkstra_path_length(self.model.G, self.home_node, weight= self.model.weight_general)

    self._cache_precomputed = True
    return self._cache_lengths

  def _get_walk_lengths(self):

    if self._cache_precomputed_walk:
      return self._cache_lengths_home_walk, self._cashe_lengths_workplace_walk

    destination_node = self.workplace_node if self.age_group == "adult" else self.school_node
    self.home_node = self.model.nearest_node(self.x, self.y)

    self._cache_lengths_home_walk = nx.single_source_dijkstra_path_length(self.model.G, self.home_node, weight= self.model.walk_distance)
    self._cashe_lengths_workplace_walk = nx.single_source_dijkstra_path_length(self.model.G, destination_node, weight= self.model.walk_distance)

    self._cache_precomputed_walk = True

    return self._cache_lengths_home_walk, self._cashe_lengths_workplace_walk

  # --------------------------------------------------------

  def initialize(self):

    # Home node
    self.home_node = self.model.nearest_node(self.x, self.y)

    # WORKPLACE (adults)
    if self.age_group == "adult":

      self.workplace_node = random.choice(self.model.workplace_nodes)

      self.dist_work_time, _ = nx.single_source_dijkstra(self.model.G, source = self.home_node, target = self.workplace_node, weight= self.model.car_time)
      self.dist_work , _ = nx.single_source_dijkstra(self.model.G, source = self.home_node, target = self.workplace_node, weight= self.model.walk_distance)

    # SCHOOL (children)
    else:
      lengths = self._get_home_lengths()

      MAX_DIST = 2 # 2 Km
      found_school = False

      # Try increasing radius
      while MAX_DIST <= 10 and not found_school:

        nearby_schools = [
            n for n in self.model.school_nodes
            if n in lengths and 0 < lengths[n] <= MAX_DIST
        ]

        if nearby_schools:
          self.school_node = random.choice(nearby_schools)
          self.dist_work_time, _ = nx.single_source_dijkstra(self.model.G, source = self.home_node, target = self.school_node, weight= self.model.car_time)
          self.dist_work , _ = nx.single_source_dijkstra(self.model.G, source = self.home_node, target = self.school_node, weight= self.model.walk_distance)

          found_school = True
        else:
          MAX_DIST += 2

      # --------------------------------------------------
      # Fallback (ALWAYS assign a school)
      # --------------------------------------------------
      if not found_school:

        self.school_node = random.choice(self.model.school_nodes)

        self.dist_work_time,_ = nx.single_source_dijkstra(self.model.G, source = self.home_node, target = self.school_node, weight= self.model.car_time)
        self.dist_work,_ = nx.single_source_dijkstra(self.model.G, source = self.home_node, target = self.school_node, weight= self.model.walk_distance)


  def Bixi_STM_settings(self): # this function is seperately designed for investigating the applied changes in the Bixi and STM settings as well


    lengths_home, lengths_workplace = self._get_walk_lengths()

    # Home-BIXI,
    reachable_bixi_home = [(n, lengths_home[n]) for n in self.model.bixi_nodes if n in lengths_home]

    if reachable_bixi_home:
      self.home_bixi_node, self.dist_home_bixi = min(reachable_bixi_home, key=lambda x: x[1])
    else:
      print("There is no reachable bixi station from home! ")

    # BIXI-Work

    reachable_bixi_workplace = [(n, lengths_workplace[n]) for n in self.model.bixi_nodes if n in lengths_workplace]

    if reachable_bixi_workplace:
      self.work_bixi_node, self.dist_bixi_work = min(reachable_bixi_workplace, key=lambda x: x[1])
    else:
      print("There is no reachable bixi station from workplace! ")

    # Bixi density in the radius 1km ------------------- For intervention
    self.bixi_density = sum(1 for n in self.model.bixi_nodes if n in lengths_home and lengths_home[n] <= 1)

    # Home-STM
    reachable_stm_home = [(n, lengths_home[n]) for n in self.model.stm_nodes if n in lengths_home]

    if reachable_stm_home:
      self.home_stm_node, self.dist_home_stm = min(reachable_stm_home, key=lambda x: x[1])
    else:
      print("There is no reachable STM station from home! ")

    # STM-Work
    reachable_stm_workplace = [(n, lengths_workplace[n]) for n in self.model.stm_nodes if n in lengths_workplace]

    if reachable_stm_workplace:
      self.work_stm_node, self.dist_stm_work = min(reachable_stm_workplace, key=lambda x: x[1])
    else:
      print("There is no reachable STM station from workplace! ")


    # Computing distances from the home-nearest Bixi and STM station to the workplace/school-nearest Bixi and STM station
    # Bixi lane ratio

    # Distance and Path
    self.dist_bixi_bixi, path = nx.single_source_dijkstra(self.model.G, self.home_bixi_node, self.work_bixi_node, weight=self.model.bike_distance)

    # Bike lane ratio
    total = 0
    bike = 0

    for u, v in zip(path[:-1], path[1:]):
      edge_data = self.model.G.get_edge_data(u, v)
      edge = next(iter(edge_data.values()))
      total += edge.get("edge_length", 0)

      if edge.get("no_bikelane", 0) == 0:
          bike += edge.get("edge_length", 0)

    self.bike_lane_ratio = bike / total if total > 0 else 0


    # Distance from home-nearest STM station to the workplace/school-nearest STM
    self.dist_stm_stm = nx.shortest_path_length(self.model.G, self.home_stm_node, self.work_stm_node, weight= self.model.weight_general)


  # --------------------------------------------------------
  def compute_utilities(self, betas, df, speed_transit = 25, speed_bike = 15, speed_walk = 5):

    (beta_time_auto,
    beta_time_transit,
    beta_time_bike,
    beta_time_walk,
    beta_income_auto,
    beta_income_transit,
    beta_bike_lane,
    beta_bixi_density,
    beta_age_bike,
    beta_age_walk,
    ASC_auto,
    ASC_transit,
    ASC_bike) = betas

    # Min-max normalization
    def safe_norm(val, series):
        denom = series.max() - series.min()
        if denom == 0 or np.isnan(denom):
            return 0
        return (val - series.min()) / denom

    distt_work = safe_norm(self.dist_work, df['dist_work'])
    distt_work_time = safe_norm(self.dist_work_time, df['dist_work_time'])
    distt_home_stm = safe_norm(self.dist_home_stm, df['dist_home_stm'])
    distt_home_bixi = safe_norm(self.dist_home_bixi, df['dist_home_bixi'])
    distt_stm_work= safe_norm(self.dist_stm_work, df['dist_stm_work'])
    distt_bixi_work = safe_norm(self.dist_bixi_work, df['dist_bixi_work'])
    distt_bixi_bixi = safe_norm(self.dist_bixi_bixi, df['dist_bixi_bixi'])
    distt_stm_stm = safe_norm(self.dist_stm_stm, df['dist_stm_stm'])
    incomee = safe_norm(self.income, df['income'])
    agee  = safe_norm(self.age, df['age'])
    bike_lane_ratioo = safe_norm(self.bike_lane_ratio, df['bike_lane_ratio'])
    bixi_densityy = safe_norm(self.bixi_density, df['bixi_density'])


    # Time takes to get to the work from home
    time_auto = distt_work_time # it is already time
    time_transit = (distt_home_stm / speed_walk) + (distt_stm_stm / speed_transit) + (distt_stm_work / speed_walk)
    time_bike = (distt_home_bixi / speed_walk) + (distt_bixi_bixi / speed_bike) + (distt_bixi_work / speed_walk)
    time_walk = distt_work / speed_walk

    # Utilities

    self.U_auto = (ASC_auto + beta_time_auto * time_auto + beta_income_auto * incomee)

    self.U_transit = (ASC_transit + beta_time_transit * time_transit + beta_income_transit * incomee)

    self.U_bike = (ASC_bike + beta_time_bike * time_bike + beta_bike_lane * bike_lane_ratioo + beta_bixi_density * bixi_densityy + beta_age_bike * agee)

    self.U_walk = (beta_time_walk * time_walk + beta_age_walk * agee)

  # --------------------------------------------------------
  def compute_probabilities(self):

    utilities = np.array([self.U_auto, self.U_transit, self.U_bike, self.U_walk])

    exp_u = np.exp(utilities)
    sum_exp = np.sum(exp_u)

    if sum_exp == 0 or np.isnan(sum_exp) or np.isinf(sum_exp):
        probs = np.ones(4) / 4
    else:
        probs = exp_u / sum_exp

    # final safety check
    if np.any(~np.isfinite(probs)) or np.any(probs < 0):
        probs = np.ones(4) / 4

    self.P_auto, self.P_transit, self.P_bike, self.P_walk = probs

    return probs

  # --------------------------------------------------------
  def choose_mode(self):
    modes = ["auto", "transit", "bike", "walk"]
    probs = [self.P_auto, self.P_transit, self.P_bike, self.P_walk]

    self.mode_choice = random.choices(modes, weights=probs, k=1)[0]

In [ ]:
def calibration_error(betas, model):

  df = model.datacollector.get_agent_vars_dataframe()

  # Update utilities and probabilities
  for a in model.agent_list:
    if a.age_group == "adult":
      a.compute_utilities(betas, df)
      a.compute_probabilities()
      a.choose_mode()

  # Simulated mode shares
  sim = model.compute_mode_shares()

  total_error = 0
  total_weight = 0

  pbar = tqdm(model.montreal_da.iterrows(), total=len(model.montreal_da))

  for _, da in pbar:
    pbar.set_description(f"Calibrating DA: {da['DA_name']}")

    if da['Car_truck_'] == 0 and da['Public_tra'] == 0 and da['Bicycle'] == 0 and da['Walked'] == 0:
      continue

    res = model.residential[model.residential.intersects(da.geometry)]
    if res.empty:
      continue


    da_id = da['DA_name']
    if da_id not in sim:
        continue

    # Observed counts
    observed_counts = np.array([int(da['Car_truck_'] / model.Each_agent_represents), int(da['Public_tra'] / model.Each_agent_represents), int(da['Bicycle'] / model.Each_agent_represents), int(da['Walked'] / model.Each_agent_represents)])

    # Simulated counts
    simulated_counts = np.array([sim[da_id]['auto'],sim[da_id]['transit'],sim[da_id]['bike'],sim[da_id]['walk']])

    obs_total = observed_counts.sum()
    sim_total = simulated_counts.sum()

    # Skip invalid cases
    if obs_total == 0 or np.isnan(obs_total):
        continue

    if sim_total == 0 or np.isnan(sim_total): # it must never return zero but just in case
       continue

    simulated = simulated_counts / sim_total
    observed = observed_counts / obs_total

    # MSE per DA
    mse_da = np.mean((observed - simulated) ** 2)

    # Weight by population
    weight = observed.sum()

    total_error += weight * mse_da
    total_weight += weight

  df_updated = model.datacollector.get_agent_vars_dataframe()
  df_updated.to_csv("/content/drive/MyDrive/Transportation_UrbanHealth/Data_collecter_results.csv", index=False)
  # Return weighted MSE
  return total_error / total_weight if total_weight > 0 else np.inf

#Considering each agent as the represener of 100 people

In [ ]:
initial_betas = np.random.normal(0, 0.1, 13)

# Initialize the transport model
Transport_model = TransportModel(n_rep=100)
Transport_model.step()

mse_history = []

def callback_fun(xk):
  mse_val = calibration_error(xk, Transport_model)
  mse_history.append(mse_val)

  print(f"MSE: {mse_val:.6f}")
  print(f"Betas: {xk}\n")

# Optimize coefficients using L-BFGS-B (MLE approach)
result = minimize(
    calibration_error,          # objective function (negative log-likelihood)
    initial_betas,              # starting values for betas
    args=(Transport_model,),    # pass the model to the objective
    method='L-BFGS-B',          # more efficient for many parameters
    callback=callback_fun,      # track progress
    options={'maxiter': 5}     # maximum iterations
)


print("Optimized Betas:", result.x)
np.save("/content/drive/MyDrive/Transportation_UrbanHealth/Betas.npy", result.x)
final_mse = calibration_error(result.x, Transport_model)
print("Final MSE:", final_mse)

#Considering each agent as the represener of 50 people

In [ ]:
initial_betas = initial_betas = np.random.normal(0, 0.1, 13)

# Initialize the transport model
Transport_model = TransportModel(n_rep=50)
Transport_model.step()

mse_history = []

def callback_fun(xk):
  mse_val = calibration_error(xk, Transport_model)
  mse_history.append(mse_val)

  print(f"MSE: {mse_val:.6f}")
  print(f"Betas: {xk}\n")

# Optimize coefficients using L-BFGS-B (MLE approach)
result = minimize(
    calibration_error,          # objective function (negative log-likelihood)
    initial_betas,              # starting values for betas
    args=(Transport_model,),    # pass the model to the objective
    method='L-BFGS-B',          # more efficient for many parameters
    callback=callback_fun,      # track progress
    options={'maxiter': 10}     # maximum iterations
)


print("Optimized Betas:", result.x)
final_mse = calibration_error(result.x, Transport_model)
print("Final MSE:", final_mse)